In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ["OPENAI_API_KEY"]:
    print("OPENAI_API_KEY is set.")

OPENAI_API_KEY is set.


In [6]:
from langchain_openai import ChatOpenAI



In [7]:
llm = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=0)

In [9]:
response = llm.invoke("What is my name")
response.content

'I don’t know your name unless you tell me. What would you like me to call you? You can share your real name or a nickname, and I’ll use it for this chat.'

## **RAG Implementation with our own text data**

### Step 1: Preparing Documents for our text

In [11]:
from langchain_core.documents import Document

In [14]:
my_text = """SATHYA RAJESH PK
Lead   |  Azure Data Engineering  |  BI & Analytics  |  AI-Powered Data Automation
Chennai, Tamil Nadu, India  |  +91-9597996996  |  sathyarajeshpk17@gmail.com  |  linkedin.com/in/sathyarajeshpk
PROFESSIONAL SUMMARY
Versatile Data Engineer and Analyst with 12+ years of IT experience, including 4 years leading cloud-based data solutions at TransUnion. Combines deep expertise in Azure data engineering (ADF, Databricks, Synapse, ADLS Gen2) with strong analytical capability in Power BI, SQL, Python, and operational KPI reporting. Skilled at bridging the gap between raw data pipelines and business-ready insights – from architecting scalable ETL/ELT workflows and Delta Lake models to delivering executive dashboards and data storytelling. Proven track record of mentoring engineering teams, driving process efficiency, and delivering AI-powered data tools that accelerate business value.
CORE TECHNICAL COMPETENCIES
•	Data Engineering: Azure Data Factory, Azure Databricks (PySpark, Spark SQL), Synapse Analytics, Azure Data Lake Storage Gen2, Delta Lake, ETL/ELT Pipeline Design, Data Modeling, Schema Design, Data Governance
•	Data Analysis & BI: Power BI (DAX, Power Query), Apache Superset, Tableau, KPI Reporting, MIS Reporting, Operational Analytics, MTTA/MTTR Tracking
•	Programming & Scripting: Python (Pandas, NumPy), SQL, PySpark, Spark SQL, Bash, Linux Administration
•	Cloud & DevOps: Azure SQL Database, Azure Key Vault, Azure Monitor, Log Analytics, GCP Bigquery(exposure), Self-hosted Integration Runtime (SHIR). 
•	Other Tools: Splunk, Autosys, AB Initio GDE, Power Query, Power Automate, Microsoft Fabric, DBT
PROFESSIONAL EXPERIENCE
TransUnion Global Technology Center LLP, Chennai  |  Lead  |  Nov 2019 – Nov 2025
•	Architected and deployed 10+ production-grade ETL/ELT pipelines using Azure Data Factory and Azure Databricks, processing millions of records daily with 99.5% SLA uptime.
•	Led migration of 50+ TB of legacy data to Azure Data Lake Storage Gen2, achieving 30% cost reduction and 40% faster query performance.
•	Implemented Delta Lake architecture with incremental load strategies (MERGE, UPSERT) for high-performance OLAP analytics and reporting.
•	Built Power BI executive dashboards and automated MIS reports for KPI tracking, and operational performance, reducing manual reporting effort by ~60%.
•	Tracked and reported operational KPIs including MTTA and MTTR using Splunk, Autosys, and Power BI, enabling data-driven SLA management and resource planning.
•	Analyzed large operational datasets using SQL and Python (Pandas) to identify workload trends, incident patterns, and performance improvement opportunities.
•	Established proactive pipeline monitoring using Azure Monitor and Log Analytics, reducing unplanned downtime and improving incident response efficiency.
•	Mentored 20+ data engineers on PySpark, Azure best practices, and data engineering standards, improving team productivity by 25%.
•	Collaborated with architecture, operations, and business stakeholders to deliver governed, compliant, and insight-ready data solutions.

TECHNICAL PROJECTS & KEY ACHIEVEMENTS
•	Power BI Executive Dashboard: Scaffolded a full Power BI executive dashboard with star schema data model, DAX measures, red/black executive theme, and synthetic data for client demonstration.
•	AI-Powered NL2SQL Tool: Designed and deployed an AI application that converts natural language questions into SQL, PySpark, and Python code with automated chart generation using Claude API.
•	HR Analytics Dashboard (Apache Superset): Built an analytics dashboard to analyze employee attrition, hiring trends, and workforce performance metrics.
•	Incident Root Cause Analyzer: Developed an AI-assisted log analysis solution to automatically identify root causes of operational failures, accelerating incident investigation.
RECOGNITION & AWARDS
•	Quarterly Best Performer Award – TransUnion (Feb 2024, Aug 2023, Nov 2022)
•	Client Appreciation Award – Tata Consultancy Services for high-quality technical delivery
•	Wall of Fame – Sitel India (3 consecutive quarters for outstanding performance)
LEADERSHIP & COMMUNITY
•	Mentor – Foundation for Excellence (FFE): Guided college students on career development, data analytics learning paths, and internship preparation.
EDUCATION
Bachelor of Engineering (B.E.) – Electrical & Electronics Engineering
Anna University, Chennai  |  Graduated: Dec 2012
"""


docs = [Document(page_content=my_text, metadata={"source":"Sathya_Resume", "documentID":"Resume"})]
docs

[Document(metadata={'source': 'Sathya_Resume', 'documentID': 'Resume'}, page_content='SATHYA RAJESH PK\nLead   |  Azure Data Engineering  |  BI & Analytics  |  AI-Powered Data Automation\nChennai, Tamil Nadu, India  |  +91-9597996996  |  sathyarajeshpk17@gmail.com  |  linkedin.com/in/sathyarajeshpk\nPROFESSIONAL SUMMARY\nVersatile Data Engineer and Analyst with 12+ years of IT experience, including 4 years leading cloud-based data solutions at TransUnion. Combines deep expertise in Azure data engineering (ADF, Databricks, Synapse, ADLS Gen2) with strong analytical capability in Power BI, SQL, Python, and operational KPI reporting. Skilled at bridging the gap between raw data pipelines and business-ready insights – from architecting scalable ETL/ELT workflows and Delta Lake models to delivering executive dashboards and data storytelling. Proven track record of mentoring engineering teams, driving process efficiency, and delivering AI-powered data tools that accelerate business value.\nC

### Step 2: Splitting document to chunks 

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

chunks = splitter.split_documents(docs)
chunks


[Document(metadata={'source': 'Sathya_Resume', 'documentID': 'Resume'}, page_content='SATHYA RAJESH PK\nLead   |  Azure Data Engineering  |  BI & Analytics  |  AI-Powered Data Automation\nChennai, Tamil Nadu, India  |  +91-9597996996  |  sathyarajeshpk17@gmail.com  |  linkedin.com/in/sathyarajeshpk\nPROFESSIONAL SUMMARY'),
 Document(metadata={'source': 'Sathya_Resume', 'documentID': 'Resume'}, page_content='Versatile Data Engineer and Analyst with 12+ years of IT experience, including 4 years leading cloud-based data solutions at TransUnion. Combines deep expertise in Azure data engineering (ADF, Databricks, Synapse, ADLS Gen2) with strong analytical capability in Power BI, SQL, Python, and operational KPI reporting. Skilled at bridging the gap between raw data pipelines and business-ready insights – from architecting scalable ETL/ELT workflows and Delta Lake models to delivering executive'),
 Document(metadata={'source': 'Sathya_Resume', 'documentID': 'Resume'}, page_content='and Delt

### Step 3: Creating Embeddings for the Chunks

In [31]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model = "text-embedding-3-small") 


### Step 4 : Using Chroma DB as vector store. Create and store embedding in the vector store

#### Langchain will take care of embedding of our each chunk and store it in vector store



In [33]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)


# In background, this happened - Langchain did for us

# vectors = []
# for doc in chunks:
#     vector = embedding_model.embed_documents([doc.page_content])
#     vectors.append(vector)

### Similarity or Semantic Search

In [40]:
vectorstore.similarity_search("Who is Sathya", k=3)

## User probes here

[Document(metadata={'documentID': 'Resume', 'source': 'Sathya_Resume'}, page_content='SATHYA RAJESH PK\nLead   |  Azure Data Engineering  |  BI & Analytics  |  AI-Powered Data Automation\nChennai, Tamil Nadu, India  |  +91-9597996996  |  sathyarajeshpk17@gmail.com  |  linkedin.com/in/sathyarajeshpk\nPROFESSIONAL SUMMARY'),
 Document(metadata={'documentID': 'Resume', 'source': 'Sathya_Resume'}, page_content='SATHYA RAJESH PK\nLead   |  Azure Data Engineering  |  BI & Analytics  |  AI-Powered Data Automation\nChennai, Tamil Nadu, India  |  +91-9597996996  |  sathyarajeshpk17@gmail.com  |  linkedin.com/in/sathyarajeshpk\nPROFESSIONAL SUMMARY'),
 Document(metadata={'documentID': 'Resume', 'source': 'Sathya_Resume'}, page_content='•\tClient Appreciation Award – Tata Consultancy Services for high-quality technical delivery\n•\tWall of Fame – Sitel India (3 consecutive quarters for outstanding performance)\nLEADERSHIP & COMMUNITY\n•\tMentor – Foundation for Excellence (FFE): Guided college st

### Talk to LLM

In [41]:
context = vectorstore.similarity_search("Who is Sathya", k=3)

In [43]:
response = llm.invoke(f"Who's Arun. Answer using {context}")
print(response.content)

I didn’t find any mention of a person named Arun in the provided documents. The materials are about Sathya Rajesh PK, including:

- Name and contact details, title: Lead | Azure Data Engineering | BI & Analytics | AI-Powered Data Automation
- Professional summary
- Awards and recognition (Client Appreciation Award – Tata Consultancy Services; Wall of Fame – Sitel India)
- Leadership & Community activities (Mentor – Foundation for Excellence)
- Education (B.E. – Electrical & Electronics Engineering, Anna University, Dec 2012)

If you intended a different document or want me to search for Arun in another file, please share it.
